<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# The Duckietown Workspace: Docker for Development

In previous notebooks we have learned how Docker can package development tools as well as applications. Instead of installing every required tool directly on your computer, you can work inside a container that provides a prepared environment.

The [__Duckietown Workspace__](https://docs.duckietown.com/ente/duckietown-manual/10-setup/00-computer/setup-duckietown-workspace.html), often used for this LX, applies this idea to Duckietown development. It includes the Duckietown Shell (`dts`), Python and Jupyter tooling, a browser-accessible desktop, and Docker-in-Docker support for virtual Duckiedrones. This provides a consistent learning and development environment across supported host systems.

In this final notebook, we will use the Docker concepts we have learned to understand the environment in which development happens.

## A development environment inside a container

A __development container__ (or devcontainer) supplies tools, libraries, and configuration for working on projects. VS Code can connect to it so that its integrated terminals and development tools operate inside that environment.

The project's `.devcontainer/devcontainer.json` describes how VS Code should open the environment. It can reference a Docker image or a Compose configuration and specify additional editor settings. 

The [Duckietown Workspace](https://github.com/duckietown/workspace) uses this mechanism to provide a prepared Linux environment. Opening a terminal inside it gives you access to the Workspace's tools and filesystem.

A terminal opened directly on your computer may therefore have different programs, files, and configuration, even when both terminals appear on the same screen.

## Docker can run inside Docker

The Workspace includes __Docker-in-Docker__: a Docker daemon running inside the development container.

[Figure 1](#figure-1) shows the two Docker daemon roles in the Workspace.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    outer Docker daemon
      |
      | start the Workspace container
      v
    Workspace container
      |
      +-> development tools: shell, dts, Python, ...
      |
      `-> inner Docker daemon
            |
            | manage an independent Docker inventory
            v
          containers started inside the Workspace
  </pre>
  <figcaption>Figure 1: The outer daemon runs the Workspace, whose inner daemon manages a separate Docker inventory.</figcaption>
</figure>

The outer daemon manages the Workspace container. The inner daemon manages a separate inventory of images and containers.

This is a configured feature of the Workspace. Installing a Docker client inside an ordinary container does not automatically give that container its own daemon.

Also, a client inside the Workspace can use a context to reach another daemon, including one on a physical Duckiedrone. The location of the client and the destination of its commands remain separate questions.

## Compare two terminals

If you have the Workspace running, open:

1. A terminal directly on your computer.

2. A terminal inside the Workspace.

If you are not using the Workspace because you are working on a native Ubuntu system, read the activity as an example; setting it up is not required to complete this LX.

Run the following in each terminal:

```bash
hostname
pwd
command -v docker
command -v dts
```

Record the differences. On macOS, `hostname` may report a name derived from the current network, including a router-provided domain suffix. It can differ from the computer name shown in System Settings and may change between networks. Use it as an environment clue, not as proof of which Docker daemon receives your commands. `command -v` reports how the shell resolves a command; no result means it did not find that command in the current environment.

Keeping `dts` installed both on your computer and inside the Workspace makes it available in both environments. The host installation supports commands from a host terminal, while the Workspace installation supports commands using its prepared development tools. Each installation uses its environment's files and configuration. Two installations do not by themselves create two Docker daemons, and you do not need both if you only use one environment.

Next, inspect Docker's connection settings:

```bash
docker context ls
printf 'DOCKER_HOST=%s\n' "${DOCKER_HOST:-<unset>}"
printf 'DOCKER_CONTEXT=%s\n' "${DOCKER_CONTEXT:-<unset>}"
```

Use the contexts notebook to identify the endpoint selected in each terminal. Then query the daemon:

```bash
docker info --format 'Daemon name: {{.Name}} | ID: {{.ID}}'
docker container ls --all
```

Compare the daemon IDs and inventories.

Different IDs indicate different daemons. Matching IDs indicate that both clients reached the same daemon, even though the shells may have different files and tools.

With the Workspace's inner daemon selected, its inventory should differ from the outer daemon's inventory. The Workspace container itself belongs to the outer daemon.

A hostname or terminal prompt alone does not establish which Docker daemon answered.

## Where are Workspace files stored?

The Workspace also provides a concrete example of persistent storage.

Open its [Compose configuration](https://github.com/duckietown/workspace/blob/baeb3649164d4cc06f7cf71fa69a5c2c87c4bc2b/.devcontainer/docker-compose.yml) and find this mount:

```yaml
volumes:
  - workspace-home:/home/ubuntu
```

Here, `workspace-home` is a named volume mounted at `/home/ubuntu` inside the Workspace. Compose may prefix the actual volume name with its project name.

Files saved under that directory live in the volume, outside the Workspace container's writable layer. Recreating the Workspace container can preserve them when the same volume is reused.

However, this does not automatically synchronize a repository inside the Workspace with another clone on your computer. They can contain different branches, commits, and uncommitted edits. 

When you are unsure which repository you are editing, check:

```bash
pwd
git rev-parse --show-toplevel
git status --short --branch
```

Persistent storage also does not protect files from edits or a Git reset performed inside that storage. Commit and back up work you want to keep.

## Which filesystem supplies a bind mount?

Recall that a bind mount's source is resolved on the __Docker daemon's host__.

If the Workspace client talks to its inner daemon, that daemon sees paths in the Workspace environment. If the client instead targets a physical Duckiedrone, a bind-mount source refers to a path on that Duckiedrone.

Docker does not automatically upload a local directory just because its path appears in a mount option. 

This explains a common failure: `ls` can find a directory in your terminal, but a Docker command targeting another daemon cannot mount that same path.

When investigating it, distinguish:

- Where the shell sees the project files.

- Which daemon receives the Docker command.

- Where that daemon looks for the mount source.

## Further reading

The [Duckietown Workspace repository](https://github.com/duckietown/workspace) contains its environment configuration. VS Code's [development-container documentation](https://code.visualstudio.com/docs/devcontainers/containers) explains how the editor connects to and works inside these environments, and Docker's [bind-mount documentation](https://docs.docker.com/engine/storage/bind-mounts/) explains how it resolves mount paths. 

## Checkpoint

Before revealing the answers, consider:

1. Why might `dts` be available inside the Workspace but absent from a terminal on your computer?

2. Why might it make sense to install two copies of `dts` on the same computer, inside and outside a Workspace?

3. Which daemon manages the Workspace container itself?

4. Does a repository inside the Workspace automatically stay synchronized with a clone outside it?

5. If your Docker client targets a physical Duckiedrone, where must a bind-mount source directory exist?

Run the self-check in the next cell.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()


## Summary and conclusions

The Duckietown Workspace combines a development container, persistent home storage, and an inner Docker daemon. Comparing terminals showed why shell files, available tools, and Docker targets must be checked separately. This completes the Docker LX: you can now relate images, containers, storage, networks, and contexts to Duckietown development. Continue with the [ROS 2 LX](https://github.com/duckietown/lx-dd-middleware-ros2) to learn how robotics applications exchange messages through middleware.
